In [21]:
import sys
from pathlib import Path
import numpy as np
#from numpy.linalg import eigvals, eigvalsh
#from scipy.linalg import eigh
from scipy.sparse.linalg import eigsh,eigs
import matplotlib.pyplot as plt
import math, cmath
import sys
import os
import joblib
from pathlib import Path
import pickle
import time
from mpl_toolkits.mplot3d import Axes3D
from scipy.sparse import csr_matrix, coo_matrix,  bmat, diags, identity, issparse, vstack, hstack



In [22]:

import sys  
sys.path.insert(1, '\\\\smb.physics.wisc.edu\\smb\\friesen-group\\archives\\avivrekar\\Topological qubit\\Codes\\Spin_Splitting_Ge_ABS')


In [23]:
import Operators2D as Op

In [24]:

hbar = 1.055*10**(-34);   #in J-s
ee = 1.602*10**(-19);     #in C
hbar_mev = hbar/ee*1000   # in meV-s
me = 9.11*10**(-31);      #in kg
hbm = hbar**2*1000/2/me/ee/(1e-18) #in meV (nm)**2

In [25]:
NSM = 300          # in nm
NSC = 2000         # in nm
N = 2*NSC + NSM
AX = 1
ALPHAT = 20        # meV-nm
DELTA = 1.65      # in meV
MEFF = 0.023       
PHI = 0.5*np.pi 
MU_const = 0.1
LY = 100
OMEGA = 2*4*hbm/LY**2/MEFF


In [26]:
KXSQ = Op.Operators_2d(N,1,AX,1).KxSq()
KX = Op.Operators_2d(N,1,AX,1).Kx()
KXCU = Op.Operators_2d(N,1,AX,1).KxCu()
MU = MU_const * Op.Operators_2d(N,1,AX,1).iden()

In [27]:
EY_minus = OMEGA/2
KAPPA = math.sqrt(OMEGA*MEFF/4/hbm)
TOTAL = math.sqrt(EY_minus**2+(ALPHAT*KAPPA)**2)
COS = EY_minus/TOTAL
SIN = ALPHAT*KAPPA/TOTAL

In [28]:
HA = hbm/MEFF*KXSQ - MU - (ALPHAT*SIN)**2*KXSQ/2/TOTAL

In [29]:
HB = ALPHAT*KX*COS - ALPHAT**3*SIN**2*COS*KXCU/2/TOTAL

In [30]:
DEL_nsc1 = np.full(NSC, DELTA)
DEL_nsc2 = np.full(NSC, DELTA*cmath.exp(1j*PHI))
DEL_nsm = np.full(NSM, 0 )

DEL_array = np.concatenate((DEL_nsc1, DEL_nsm, DEL_nsc2))

DEL = diags(DEL_array)

In [32]:
H = bmat([
    [HA + HB, None, None, DEL],
    [None, HA-HB, -DEL, None],
    [None,-coo_matrix.conjugate(DEL), -HA+HB, None],
    [coo_matrix.conjugate(DEL), None, None, -HA-HB]
])

In [33]:
E = eigsh(H,k=8,sigma=0, return_eigenvectors=False)

In [34]:
Es = np.sort(E)

In [35]:
print(Es)

[-0.34936681 -0.34928628 -0.0299265  -0.02986592  0.02986592  0.0299265   0.34928628  0.34936681]
